# HW4 — Image Restoration with PromptIR
### Visual Recognition using Deep Learning, Spring 2026

> **Prerequisites**: Make sure you've switched the runtime to **GPU** (T4)!
> Runtime → Change runtime type → GPU


## Step 0: Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Step 1: Mount Google Drive (Recommended for saving checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Change this to your preferred directory on Drive
SAVE_DIR = '/content/drive/MyDrive/HW4_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Save directory:', SAVE_DIR)

## Step 2: Install Dependencies

In [ ]:
!pip install einops tqdm -q

## Step 3: Clone / Upload Your Code

**Option A**: Upload your code zip to Colab and unzip.
**Option B**: Clone from your GitHub repo (replace URL).

In [ ]:
# Option A: Upload zip
# from google.colab import files
# uploaded = files.upload()  # upload hw4_code.zip
# !unzip hw4_code.zip -d /content/hw4

# Option B: Clone from GitHub
# !git clone https://github.com/YOUR_ID/YOUR_REPO.git /content/hw4

# For now, we write code inline:
os.makedirs('/content/hw4/net', exist_ok=True)
os.makedirs('/content/hw4/utils', exist_ok=True)
os.makedirs('/content/hw4/checkpoints', exist_ok=True)
%cd /content/hw4
print('Working directory:', os.getcwd())

## Step 4: Write Model Code

In [ ]:
%%writefile /content/hw4/net/__init__.py


In [ ]:
%%writefile /content/hw4/net/model.py
"""
PromptIR: Prompting for All-in-One Blind Image Restoration (NeurIPS 2023)
Adapted for Rain/Snow restoration task.

Modifications from baseline:
1. Added Channel Attention in each Transformer block (SE-block style)
2. Added auxiliary FFT-based frequency loss support (used in train.py)
3. Increased prompt length from 5 to 10 for richer degradation encoding
4. Added residual scaling for more stable training
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import numbers


def to_3d(x):
    return rearrange(x, 'b c h w -> b (h w) c')

def to_4d(x, h, w):
    return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)

class BiasFree_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.normalized_shape = normalized_shape
    def forward(self, x):
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma + 1e-5) * self.weight

class WithBias_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias   = nn.Parameter(torch.zeros(normalized_shape))
        self.normalized_shape = normalized_shape
    def forward(self, x):
        mu    = x.mean(-1, keepdim=True)
        sigma = x.var(-1,  keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma + 1e-5) * self.weight + self.bias

class LayerNorm(nn.Module):
    def __init__(self, dim, LayerNorm_type='WithBias'):
        super().__init__()
        self.body = BiasFree_LayerNorm(dim) if LayerNorm_type == 'BiasFree' else WithBias_LayerNorm(dim)
    def forward(self, x):
        h, w = x.shape[-2:]
        return to_4d(self.body(to_3d(x)), h, w)

class FeedForward(nn.Module):
    def __init__(self, dim, ffn_expansion_factor=2.66, bias=False):
        super().__init__()
        hidden = int(dim * ffn_expansion_factor)
        self.project_in  = nn.Conv2d(dim, hidden * 2, 1, bias=bias)
        self.dwconv      = nn.Conv2d(hidden * 2, hidden * 2, 3, 1, 1, groups=hidden * 2, bias=bias)
        self.project_out = nn.Conv2d(hidden, dim, 1, bias=bias)
    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        return self.project_out(F.gelu(x1) * x2)

class Attention(nn.Module):
    def __init__(self, dim, num_heads, bias=False):
        super().__init__()
        self.num_heads   = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv         = nn.Conv2d(dim, dim * 3, 1, bias=bias)
        self.qkv_dwconv  = nn.Conv2d(dim * 3, dim * 3, 3, 1, 1, groups=dim * 3, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, 1, bias=bias)
    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)
        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        out  = attn @ v
        out  = rearrange(out, 'b head c (h w) -> b (head c) h w', h=h, w=w)
        return self.project_out(out)

class ChannelAttention(nn.Module):
    """SE-block style channel attention (Modification #1)."""
    def __init__(self, dim, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(dim, dim // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // reduction, dim, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, ffn_expansion_factor=2.66, bias=False,
                 LayerNorm_type='WithBias', use_ca=True):
        super().__init__()
        self.norm1 = LayerNorm(dim, LayerNorm_type)
        self.attn  = Attention(dim, num_heads, bias)
        self.norm2 = LayerNorm(dim, LayerNorm_type)
        self.ffn   = FeedForward(dim, ffn_expansion_factor, bias)
        self.ca    = ChannelAttention(dim) if use_ca else nn.Identity()
        self.scale = nn.Parameter(torch.ones(1) * 0.1)  # Modification #4
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.scale * self.ca(self.ffn(self.norm2(x)))
        return x

class OverlapPatchEmbed(nn.Module):
    def __init__(self, in_c=3, embed_dim=48, bias=False):
        super().__init__()
        self.proj = nn.Conv2d(in_c, embed_dim, 3, 1, 1, bias=bias)
    def forward(self, x):
        return self.proj(x)

class Downsample(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feat, n_feat // 2, 3, 1, 1, bias=False),
            nn.PixelUnshuffle(2),
        )
    def forward(self, x):
        return self.body(x)

class Upsample(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feat, n_feat * 2, 3, 1, 1, bias=False),
            nn.PixelShuffle(2),
        )
    def forward(self, x):
        return self.body(x)

class PromptBlock(nn.Module):
    """Prompt block — Modification #3: prompt_len=10."""
    def __init__(self, prompt_dim=128, prompt_len=10, prompt_size=96, lin_dim=192):
        super().__init__()
        self.prompt_param = nn.Parameter(
            torch.rand(1, prompt_len, prompt_dim, prompt_size, prompt_size)
        )
        self.linear_layer = nn.Linear(lin_dim, prompt_len)
        self.conv3x3      = nn.Conv2d(prompt_dim, prompt_dim, 3, 1, 1, bias=False)
    def forward(self, x):
        B, C, H, W = x.shape
        emb = x.mean(dim=(-2, -1))
        prompt_weights = F.softmax(self.linear_layer(emb), dim=1)
        prompt = prompt_weights.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1) * self.prompt_param
        prompt = prompt.sum(dim=1)
        prompt = F.interpolate(prompt, (H, W), mode='bilinear', align_corners=False)
        return self.conv3x3(prompt)

class PromptIR(nn.Module):
    def __init__(self, inp_channels=3, out_channels=3, dim=48,
                 num_blocks=(4, 6, 6, 8), num_refinement_blocks=4,
                 heads=(1, 2, 4, 8), ffn_expansion_factor=2.66,
                 bias=False, LayerNorm_type='WithBias',
                 prompt=True, prompt_len=10):
        super().__init__()
        self.patch_embed = OverlapPatchEmbed(inp_channels, dim)
        self.encoder_l1  = nn.Sequential(*[TransformerBlock(dim,     heads[0], ffn_expansion_factor, bias, LayerNorm_type) for _ in range(num_blocks[0])])
        self.down12       = Downsample(dim)
        self.encoder_l2  = nn.Sequential(*[TransformerBlock(dim*2,   heads[1], ffn_expansion_factor, bias, LayerNorm_type) for _ in range(num_blocks[1])])
        self.down23       = Downsample(dim*2)
        self.encoder_l3  = nn.Sequential(*[TransformerBlock(dim*4,   heads[2], ffn_expansion_factor, bias, LayerNorm_type) for _ in range(num_blocks[2])])
        self.down34       = Downsample(dim*4)
        self.latent       = nn.Sequential(*[TransformerBlock(dim*8,   heads[3], ffn_expansion_factor, bias, LayerNorm_type) for _ in range(num_blocks[3])])

        self.prompt = prompt
        if prompt:
            self.prompt3           = PromptBlock(dim*4, prompt_len, 96, dim*4)
            self.prompt2           = PromptBlock(dim*2, prompt_len, 96, dim*2)
            self.prompt1           = PromptBlock(dim,   prompt_len, 96, dim)
            self.noise_level3      = TransformerBlock(dim*8, heads[3], ffn_expansion_factor, bias, LayerNorm_type)
            self.reduce_noise_level3 = nn.Conv2d(dim*8, dim*4, 1, bias=False)
            self.noise_level2      = TransformerBlock(dim*4, heads[2], ffn_expansion_factor, bias, LayerNorm_type)
            self.reduce_noise_level2 = nn.Conv2d(dim*4, dim*2, 1, bias=False)
            self.noise_level1      = TransformerBlock(dim*2, heads[1], ffn_expansion_factor, bias, LayerNorm_type)
            self.reduce_noise_level1 = nn.Conv2d(dim*2, dim,   1, bias=False)

        self.up43            = Upsample(dim*8)
        self.reduce_chan_l3  = nn.Conv2d(dim*8, dim*4, 1, bias=False)
        self.decoder_l3      = nn.Sequential(*[TransformerBlock(dim*4, heads[2], ffn_expansion_factor, bias, LayerNorm_type) for _ in range(num_blocks[2])])
        self.up32            = Upsample(dim*4)
        self.reduce_chan_l2  = nn.Conv2d(dim*4, dim*2, 1, bias=False)
        self.decoder_l2      = nn.Sequential(*[TransformerBlock(dim*2, heads[1], ffn_expansion_factor, bias, LayerNorm_type) for _ in range(num_blocks[1])])
        self.up21            = Upsample(dim*2)
        self.reduce_chan_l1  = nn.Conv2d(dim*2, dim,   1, bias=False)
        self.decoder_l1      = nn.Sequential(*[TransformerBlock(dim,   heads[0], ffn_expansion_factor, bias, LayerNorm_type) for _ in range(num_blocks[0])])
        self.refinement      = nn.Sequential(*[TransformerBlock(dim,   heads[0], ffn_expansion_factor, bias, LayerNorm_type) for _ in range(num_refinement_blocks)])
        self.output          = nn.Conv2d(dim, out_channels, 3, 1, 1, bias=False)

    def forward(self, inp_img):
        inp_enc_l1 = self.patch_embed(inp_img)
        out_enc_l1 = self.encoder_l1(inp_enc_l1)
        inp_enc_l2 = self.down12(out_enc_l1)
        out_enc_l2 = self.encoder_l2(inp_enc_l2)
        inp_enc_l3 = self.down23(out_enc_l2)
        out_enc_l3 = self.encoder_l3(inp_enc_l3)
        inp_enc_l4 = self.down34(out_enc_l3)
        latent     = self.latent(inp_enc_l4)

        if self.prompt:
            dec3_param = self.prompt3(out_enc_l3)
            latent = torch.cat([latent, self.noise_level3(
                torch.cat([latent, F.interpolate(dec3_param, size=latent.shape[-2:], mode='bilinear', align_corners=False)], dim=1)
            )], dim=1)
            latent     = self.up43(latent)
            inp_dec_l3 = self.reduce_chan_l3(torch.cat([latent, out_enc_l3], dim=1)) + dec3_param
            out_dec_l3 = self.decoder_l3(inp_dec_l3)

            dec2_param = self.prompt2(out_enc_l2)
            out_dec_l3 = torch.cat([out_dec_l3, self.noise_level2(
                torch.cat([out_dec_l3, F.interpolate(dec2_param, size=out_dec_l3.shape[-2:], mode='bilinear', align_corners=False)], dim=1)
            )], dim=1)
            out_dec_l3 = self.up32(out_dec_l3)
            inp_dec_l2 = self.reduce_chan_l2(torch.cat([out_dec_l3, out_enc_l2], dim=1)) + dec2_param
            out_dec_l2 = self.decoder_l2(inp_dec_l2)

            dec1_param = self.prompt1(out_enc_l1)
            out_dec_l2 = torch.cat([out_dec_l2, self.noise_level1(
                torch.cat([out_dec_l2, F.interpolate(dec1_param, size=out_dec_l2.shape[-2:], mode='bilinear', align_corners=False)], dim=1)
            )], dim=1)
            out_dec_l2 = self.up21(out_dec_l2)
            inp_dec_l1 = self.reduce_chan_l1(torch.cat([out_dec_l2, out_enc_l1], dim=1)) + dec1_param
            out_dec_l1 = self.decoder_l1(inp_dec_l1)
        else:
            latent     = self.up43(latent)
            inp_dec_l3 = self.reduce_chan_l3(torch.cat([latent, out_enc_l3], dim=1))
            out_dec_l3 = self.decoder_l3(inp_dec_l3)
            out_dec_l3 = self.up32(out_dec_l3)
            inp_dec_l2 = self.reduce_chan_l2(torch.cat([out_dec_l3, out_enc_l2], dim=1))
            out_dec_l2 = self.decoder_l2(inp_dec_l2)
            out_dec_l2 = self.up21(out_dec_l2)
            inp_dec_l1 = self.reduce_chan_l1(torch.cat([out_dec_l2, out_enc_l1], dim=1))
            out_dec_l1 = self.decoder_l1(inp_dec_l1)

        return self.output(self.refinement(out_dec_l1)) + inp_img


In [ ]:
%%writefile /content/hw4/utils/__init__.py


In [ ]:
%%writefile /content/hw4/utils/losses.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class FFTFrequencyLoss(nn.Module):
    def forward(self, pred, target):
        pred_fft   = torch.fft.fft2(pred,   norm='ortho')
        target_fft = torch.fft.fft2(target, norm='ortho')
        return F.l1_loss(torch.abs(pred_fft), torch.abs(target_fft))

class CombinedLoss(nn.Module):
    def __init__(self, lambda_fft=0.05):
        super().__init__()
        self.l1  = nn.L1Loss()
        self.fft = FFTFrequencyLoss()
        self.lambda_fft = lambda_fft
    def forward(self, pred, target):
        l1_loss  = self.l1(pred, target)
        fft_loss = self.fft(pred, target)
        return l1_loss + self.lambda_fft * fft_loss, l1_loss, fft_loss


In [ ]:
%%writefile /content/hw4/utils/metrics.py
import torch, math
def compute_psnr(pred, target, max_val=1.0):
    pred   = pred.clamp(0, max_val)
    target = target.clamp(0, max_val)
    mse = torch.mean((pred - target) ** 2).item()
    return 100.0 if mse == 0 else 10 * math.log10(max_val ** 2 / mse)


In [ ]:
%%writefile /content/hw4/dataset.py
import os, random
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class RestoreTrainDataset(Dataset):
    def __init__(self, data_root, patch_size=128):
        self.patch_size = patch_size
        self.pairs = []
        degraded_dir = Path(data_root) / 'train' / 'degraded'
        clean_dir    = Path(data_root) / 'train' / 'clean'
        for i in range(1, 1601):
            for prefix in ['rain', 'snow']:
                clean_prefix = prefix + '_clean'
                deg = degraded_dir / f'{prefix}-{i}.png'
                cln = clean_dir    / f'{clean_prefix}-{i}.png'
                if deg.exists() and cln.exists():
                    self.pairs.append((str(deg), str(cln)))
        assert len(self.pairs) > 0, f'No pairs found in {data_root}'
        print(f'[Dataset] {len(self.pairs)} training pairs')

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        deg_path, cln_path = self.pairs[idx]
        deg = Image.open(deg_path).convert('RGB')
        cln = Image.open(cln_path).convert('RGB')
        i, j, h, w = self._random_crop(deg, self.patch_size)
        deg = TF.crop(deg, i, j, h, w)
        cln = TF.crop(cln, i, j, h, w)
        if random.random() > 0.5: deg, cln = TF.hflip(deg), TF.hflip(cln)
        if random.random() > 0.5: deg, cln = TF.vflip(deg), TF.vflip(cln)
        angle = random.choice([0, 90, 180, 270])
        if angle: deg, cln = TF.rotate(deg, angle), TF.rotate(cln, angle)
        return TF.to_tensor(deg), TF.to_tensor(cln)

    @staticmethod
    def _random_crop(img, s):
        w, h = img.size
        if w < s or h < s: return 0, 0, h, w
        return random.randint(0, h-s), random.randint(0, w-s), s, s

class RestoreValDataset(Dataset):
    def __init__(self, data_root, patch_size=128, val_ratio=0.05):
        train = RestoreTrainDataset(data_root, patch_size)
        n = max(1, int(len(train.pairs) * val_ratio))
        self.pairs = train.pairs[-n:]
        self.patch_size = patch_size
        print(f'[Dataset] {len(self.pairs)} val pairs')
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        d, c = self.pairs[idx]
        d = TF.center_crop(Image.open(d).convert('RGB'), self.patch_size)
        c = TF.center_crop(Image.open(c).convert('RGB'), self.patch_size)
        return TF.to_tensor(d), TF.to_tensor(c)

class RestoreTestDataset(Dataset):
    def __init__(self, data_root):
        test_dir = Path(data_root) / 'test' / 'degraded'
        self.image_paths = sorted(test_dir.glob('*.png'), key=lambda p: int(p.stem))
        assert len(self.image_paths) > 0
        print(f'[Dataset] {len(self.image_paths)} test images')
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        p = self.image_paths[idx]
        return TF.to_tensor(Image.open(p).convert('RGB')), p.name


## Step 5: Upload & Prepare Dataset

Download the dataset from the HW4 link, then upload or mount it.

In [ ]:
# ---- Option A: Upload your zip from local machine ----
# from google.colab import files
# uploaded = files.upload()   # upload hw4_data.zip
# !unzip -q hw4_data.zip -d /content/hw4/data

# ---- Option B: Download from a shared link (replace URL) ----
# !gdown 'YOUR_GOOGLE_DRIVE_FILE_ID' -O /content/hw4_data.zip
# !unzip -q /content/hw4_data.zip -d /content/hw4/data

# Verify structure
DATA_ROOT = '/content/hw4/data'
!ls {DATA_ROOT}/train/degraded | head -5
!ls {DATA_ROOT}/train/clean    | head -5
!ls {DATA_ROOT}/test/degraded  | head -5

## Step 6: Train

In [ ]:
import os, time, math, random
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import sys
sys.path.insert(0, '/content/hw4')

from dataset import RestoreTrainDataset, RestoreValDataset
from net.model import PromptIR
from utils.losses import CombinedLoss
from utils.metrics import compute_psnr

# ---- Config ----
DATA_ROOT  = '/content/hw4/data'
SAVE_DIR   = SAVE_DIR  # from Step 1 (Google Drive)
EPOCHS     = 150
BATCH_SIZE = 2          # Use 2 for T4 (16GB), 4 for A100
PATCH_SIZE = 128
LR         = 2e-4
LAMBDA_FFT = 0.05
VAL_FREQ   = 5
RESUME     = None       # Set to checkpoint path to resume

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

train_dataset = RestoreTrainDataset(DATA_ROOT, PATCH_SIZE)
val_dataset   = RestoreValDataset(DATA_ROOT, PATCH_SIZE)
train_loader  = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader    = DataLoader(val_dataset,   1,          shuffle=False, num_workers=2, pin_memory=True)

model = PromptIR(prompt=True, prompt_len=10).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

criterion = CombinedLoss(LAMBDA_FFT)
optimizer = optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.999), weight_decay=1e-4)
warmup    = 10
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS-warmup, eta_min=1e-6)

start_epoch, best_psnr = 1, 0.0
if RESUME and os.path.isfile(RESUME):
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_psnr   = ckpt.get('best_psnr', 0.0)
    print(f'Resumed from epoch {ckpt["epoch"]}, PSNR={best_psnr:.2f}')

scaler = torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))

for epoch in range(start_epoch, EPOCHS+1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    if epoch <= warmup:
        for pg in optimizer.param_groups:
            pg['lr'] = LR * epoch / warmup

    for step, (deg, cln) in enumerate(train_loader, 1):
        deg, cln = deg.to(device), cln.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
            out = model(deg)
            loss, l1, fft = criterion(out, cln)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.01)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    if epoch > warmup:
        scheduler.step()

    avg = epoch_loss / len(train_loader)
    print(f'Epoch [{epoch}/{EPOCHS}] Loss={avg:.4f} | LR={optimizer.param_groups[0]["lr"]:.2e} | {time.time()-t0:.0f}s')

    if epoch % VAL_FREQ == 0 or epoch == EPOCHS:
        model.eval()
        with torch.no_grad():
            psnr = sum(compute_psnr(model(d.to(device)).clamp(0,1), c.to(device))
                       for d, c in val_loader) / len(val_loader)
        print(f'  [Val] PSNR={psnr:.2f} dB')
        if psnr > best_psnr:
            best_psnr = psnr
            torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'best_psnr':best_psnr},
                       os.path.join(SAVE_DIR, 'best_model.pth'))
            print(f'  ✓ Best model saved! PSNR={best_psnr:.2f}')
        torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'best_psnr':best_psnr},
                   os.path.join(SAVE_DIR, 'latest_model.pth'))

print(f'Done! Best PSNR: {best_psnr:.2f} dB')

## Step 7: Inference & Generate pred.npz

In [ ]:
import numpy as np
from dataset import RestoreTestDataset
from torch.utils.data import DataLoader
import torch.nn.functional as F

CHECKPOINT = os.path.join(SAVE_DIR, 'best_model.pth')
OUTPUT_NPZ = '/content/pred.npz'
TILE_SIZE  = 256   # 0 to disable tiling
TILE_OVERLAP = 32

# Load best model
ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')

test_dataset = RestoreTestDataset(DATA_ROOT)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

def tile_infer(model, img, tile, overlap, device):
    _, C, H, W = img.shape
    if tile <= 0 or (H <= tile and W <= tile):
        with torch.no_grad():
            return model(img.to(device)).clamp(0,1).cpu()
    stride = tile - overlap
    out   = torch.zeros_like(img)
    count = torch.zeros(1,1,H,W)
    ys = sorted(set(list(range(0, H-tile+1, stride)) + ([H-tile] if H>tile else [])))
    xs = sorted(set(list(range(0, W-tile+1, stride)) + ([W-tile] if W>tile else [])))
    for y in ys:
        for x in xs:
            t = img[:,:,y:y+tile,x:x+tile].to(device)
            with torch.no_grad():
                o = model(t).clamp(0,1).cpu()
            out[:,:,y:y+tile,x:x+tile]   += o
            count[:,:,y:y+tile,x:x+tile] += 1
    return out / count.clamp(min=1)

results = {}
for img_tensor, (name,) in test_loader:
    out = tile_infer(model, img_tensor, TILE_SIZE, TILE_OVERLAP, device)
    results[name] = (out.squeeze(0).numpy() * 255).clip(0,255).astype(np.uint8)
    print(f'  {name}: {results[name].shape}')

np.savez(OUTPUT_NPZ, **results)
print(f'\nSaved to {OUTPUT_NPZ}')
print('Sample shape:', next(iter(results.values())).shape)

## Step 8: Verify pred.npz & Download

In [ ]:
# Verify
data = np.load(OUTPUT_NPZ)
print('Keys:', list(data.keys())[:10])
print('Total images:', len(data.files))
print('Shape example:', data['0.png'].shape)
print('dtype:', data['0.png'].dtype)
assert data['0.png'].shape[0] == 3, 'Should be (3, H, W)'
print('\n✓ pred.npz is valid!')

In [ ]:
# Download pred.npz to local machine
from google.colab import files
files.download(OUTPUT_NPZ)

## Step 9: Visualize Results (Optional)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as TF

# Show 4 test results
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
test_dir = Path(DATA_ROOT) / 'test' / 'degraded'
pred_data = np.load(OUTPUT_NPZ)

for col, fname in enumerate(['0.png','1.png','2.png','3.png']):
    # Input
    inp = np.array(Image.open(test_dir / fname).convert('RGB'))
    axes[0, col].imshow(inp)
    axes[0, col].set_title(f'Degraded: {fname}')
    axes[0, col].axis('off')
    # Output
    out = pred_data[fname].transpose(1, 2, 0)  # (H,W,3)
    axes[1, col].imshow(out)
    axes[1, col].set_title(f'Restored: {fname}')
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('/content/results_preview.png', dpi=150)
plt.show()
print('Preview saved.')